#Logistic Regression Model Deployment

In [52]:
%%writefile preprocessing.py
import re
from nltk.tokenize import word_tokenize
from nltk.stem import WordNetLemmatizer

wnl = WordNetLemmatizer()


def preprocess_text(text: str) -> str:
    text = re.sub(r'[^a-zA-Z0-9]', ' ', text)
    text = text.lower()
    text = word_tokenize(text)
    text = [wnl.lemmatize(word) for word in text]
    text = ' '.join(text)
    return text

Writing preprocessing.py


#Importing the Libraries

In [53]:
import numpy as np
import pandas as pd

In [54]:
import nltk

In [55]:
nltk.download('punkt')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


True

In [56]:
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Package punkt_tab is already up-to-date!


True

In [57]:
nltk.download('wordnet')

[nltk_data] Downloading package wordnet to /root/nltk_data...
[nltk_data]   Package wordnet is already up-to-date!


True

In [58]:
nltk.download('omw-1.4')

[nltk_data] Downloading package omw-1.4 to /root/nltk_data...
[nltk_data]   Package omw-1.4 is already up-to-date!


True

#Importing the Dataset

In [59]:
dataset = pd.read_csv("Restaurant_Reviews.tsv", delimiter="\t", quoting=3);

In [60]:
dataset.drop_duplicates(inplace=True)

#Splitting the Dataset into the Training Set and Test Set

In [61]:
from sklearn.model_selection import train_test_split;

x_train , x_test , y_train , y_test = train_test_split(dataset, dataset.iloc[:,-1], test_size=0.2, random_state=42);

#Dataset Cleaning

In [62]:
x_train["Review"]=x_train["Review"].str.strip()

In [63]:
x_train['Words']=x_train["Review"].str.split().apply(len)

In [64]:
x_train=x_train[x_train["Words"]<28.5]

In [65]:
x_test["Review"]=x_test["Review"].str.strip()

In [66]:
y_train=x_train.loc[:,"Liked"].values

#Cleaning the Text

In [67]:
from preprocessing import preprocess_text

#Model Training

In [68]:
from sklearn.feature_extraction.text import TfidfVectorizer;
from sklearn.linear_model import LogisticRegression

In [69]:
vec=TfidfVectorizer(max_features=400, preprocessor=preprocess_text)

In [70]:
X_train=vec.fit_transform(x_train.iloc[:,0]);

In [71]:
model= LogisticRegression()

In [72]:
model.fit(X_train,y_train)

LogisticRegression()

In [73]:
model.predict_proba(vec.transform(["Food was Amazing. Also all things are served on the time. Waiter service is satisfactory"]))*100

array([[27.89829193, 72.10170807]])

In [74]:
model.predict(vec.transform(["Food was Amazing. Also all things are served on the time. Waiter service is satisfactory"]))

array([1])

In [75]:
import pickle

model_data = {
    "vectorizer": vec,
    "classifier": model,
}

with open("sentiment_analysis_model.pkl", "wb") as file:
    pickle.dump(model_data, file)